# MLB Feature Exploration (FanGraphs-free stack)

**Status:** FanGraphs has blocked pybaseball scraping. This notebook routes around it by sourcing the same signal from Statcast (Baseball Savant), Baseball-Reference, and MLB-StatsAPI, computing FG-equivalent metrics ourselves where needed.

## Working data sources

| Source | Granularity | What we get from it |
|---|---|---|
| **Baseball Savant (Statcast)** | Pitch-level | velocity, spin, movement, xwOBA, Barrel%, HardHit%, exit velo, launch angle, per-pitch `delta_run_exp` |
| **Baseball-Reference** | Season + date-range, per player and per team | IP, K, BB, HBP, HR, ERA, WHIP, AB, H, AVG, OBP, SLG — raw counts needed for derived metrics |
| **MLB-StatsAPI** | Game-level | schedules, probable pitchers, lineups, venue, scores |
| **Computed (`src/features/sabermetrics.py`)** | Derived | FIP, xFIP, SIERA, pythagorean win prob, EV from American odds |

## Target feature set (the modeling plan)

**Hitters** (weighted by lineup spot via PA-share):
- wRC+ (computed from wOBA + park-adjusted league constants)
- Barrel%
- PullAir% (pull-side fly-ball + line-drive rate)
- Statcast pitch run values per 100, vs each pitch type

**Starting pitcher** (weighted by expected IP / 9; bullpen gets the rest):
- SIERA (preferred) or xFIP
- Barrel% allowed
- GB / LD / FB rates (modulated by team OAA)
- Pitch mix + per-pitch run values

**Game context:** park, weather, rest, defense (OAA infield/outfield).

The bottom-up flow:

```
home_lineup * away_pitching_staff * away_defense * context  ->  home projected runs
away_lineup * home_pitching_staff * home_defense * context  ->  away projected runs
                pythagorean_win_prob(home_runs, away_runs)  ->  P(home wins)
                                P vs bookmaker price        ->  EV per bet
```

## Avoiding lookahead bias

Two safeguards built into the pipeline:

1. The Lambda snapshots **odds and StatsAPI data 4x/day**, so as we collect we accrue point-in-time history.
2. Every pull saved via `save_raw(...)` has a UTC timestamp in the filename. Joins always use the *as-of* snapshot, not the current state.

Don't try to backfill historical pre-FG-blackout features from FG-equivalent sources — that's where the leak is.


In [1]:
"""Setup: imports, cache, display options."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import pybaseball as pb
from src.features.pybaseball_utils import enable_cache, save_raw, load_raw, list_raw, show

enable_cache()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

print(f"pybaseball {pb.__version__}, pandas {pd.__version__}")
print(f"repo root: {REPO_ROOT}")
print(f"saved pulls so far: {len(list_raw())}")


pybaseball 2.2.7, pandas 3.0.3
repo root: /Users/danielmatten/mlb-ev
saved pulls so far: 8


---

## 1. Player ID lookups

Most pybaseball functions take a player ID. Each provider uses a different one:

| ID | Used by |
|---|---|
| `key_mlbam` | Baseball Savant / Statcast |
| `key_fangraphs` | FanGraphs (kept for completeness even though FG is blocked) |
| `key_bbref` | Baseball-Reference |
| `key_retro` | Retrosheet |

`playerid_lookup` is the bridge. You'll use it whenever you need to drill into a specific player's stats.


In [2]:
judge = pb.playerid_lookup("Judge", "Aaron")
ohtani = pb.playerid_lookup("Ohtani", "Shohei")

pd.concat([judge, ohtani])


,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last
0,judge,aaron,592450,judga001,judgeaa01,15640,2016.0,2026.0
0,ohtani,shohei,660271,ohtas001,ohtansh01,19755,2018.0,2026.0


---

## 2. Season-level pitching counts (Baseball-Reference)

`pitching_stats_bref(season)` is the FG-free substitute for `pitching_stats`. Returns one row per pitcher with: `IP, G, GS, W, L, ER, R, H, BB, SO, HBP, HR, BF, ERA, WHIP, FIP, mlbID` and similar.

BR's columns don't include xFIP, SIERA, or wRC+ — we compute those ourselves in Section 8 from the raw counts here.

**Sample-size sanity:** filter `IP >= 30` for relievers, `GS >= 10` for starters before treating any rate as stable.


In [3]:
pitching_2025 = pb.pitching_stats_bref(2025)
show(pitching_2025, n=3)


shape=(873, 41)  columns=41


,Name,Age,#days,Lev,Tm,G,GS,W,L,SV,IP,H,R,ER,BB,SO,HR,HBP,ERA,AB,2B,3B,IBB,GDP,SF,SB,CS,PO,BF,Pit,Str,StL,StS,GB/FB,LD,PU,WHIP,BAbip,SO9,SO/W,mlbID
1,Andrew Abbott,26,226,Maj-NL,Cincinnati,29,29,10.0,7.0,NaN,166.1,148,60,53,43,149,19,3,2.87,631,29,1,0,10,4,12,1,2,684,2677,0.66,0.16,0.11,0.33,0.24,0.11,1.148,0.276,8.1,3.47,671096
2,Mick Abel,23,226,"Maj-AL,Maj-NL","Minnesota,Philadelphia",10,8,3.0,4.0,NaN,39.0,43,28,27,16,39,8,1,6.23,157,2,2,0,3,0,2,1,0,174,720,0.64,0.15,0.11,0.37,0.25,0.04,1.513,0.318,9.0,2.44,690953
3,Philip Abner,23,225,Maj-NL,Arizona,5,0,NaN,NaN,NaN,3.2,5,2,2,3,5,0,0,4.91,15,2,0,0,1,0,0,0,0,18,71,0.63,0.17,0.11,0.30,0.10,0.30,2.182,0.500,12.3,1.67,691769


In [4]:
# Starters with a meaningful workload, sorted by ERA. ERA is noisy but useful
# as a sanity-check column; the real ranking happens after we compute
# FIP/xFIP/SIERA from the raw counts in Section 8.
#
# `pitching_stats_bref` does NOT return a `FIP` column — only the raw counts
# (SO, BB, HBP, HR, IP, ERA, WHIP). xFIP/SIERA aren't there either. We use
# the defensive `present = [c for c in cols if c in df.columns]` pattern so a
# silently-renamed column surfaces as a warning instead of crashing.

starters = pitching_2025[pitching_2025["GS"] >= 10].copy()

cols = ["Name", "Tm", "Age", "IP", "GS", "W", "L", "ERA", "WHIP", "SO", "BB", "HR"]
present = [c for c in cols if c in starters.columns]
missing = [c for c in cols if c not in starters.columns]
if missing:
    print(f"NOTE: columns not in pitching_stats_bref output: {missing}")
    print(f"available columns: {sorted(starters.columns)}")

starters[present].sort_values("ERA").head(15)


,Name,Tm,Age,IP,GS,W,L,ERA,WHIP,SO,BB,HR
219,Nathan Eovaldi,Texas,35,130.0,22,11.0,3.0,1.73,0.854,129,21,10
684,Trevor Rogers,Baltimore,27,109.2,18,9.0,3.0,1.81,0.903,103,29,6
747,Paul Skenes,Pittsburgh,23,187.2,32,10.0,10.0,1.97,0.948,216,42,11
748,Tarik Skubal,Detroit,28,216.0,34,14.0,6.0,2.17,0.870,277,37,20
477,Tyler Mahle,Texas,30,86.2,16,6.0,4.0,2.18,1.131,66,29,5
893,Yoshinobu Yamamoto,Los Angeles,26,211.0,35,17.0,9.0,2.30,0.953,234,65,16
101,Hunter Brown,Houston,26,185.1,31,12.0,9.0,2.43,1.025,206,57,17
705,Cristopher S\xc3\xa1nchez,Philadelphia,28,214.0,34,13.0,5.0,2.48,1.061,225,47,12
167,Garrett Crochet,Boston,26,213.0,33,19.0,5.0,2.54,1.009,266,46,25
108,Kris Bubic,Kansas City,27,116.1,20,8.0,7.0,2.55,1.178,116,39,6


---

## 3. Season-level batting counts (Baseball-Reference)

`batting_stats_bref(season)` is the hitter analog. Returns one row per hitter with: `PA, AB, R, H, 2B, 3B, HR, RBI, SB, BB, SO, AVG, OBP, SLG, OPS, OPS+, mlbID` and similar.

Statcast-derived metrics (xwOBA, Barrel%, HardHit%, PullAir%) are NOT in BR. You'll compute those per-hitter from the `statcast_batter` data in Section 5, then join on `mlbID`.


In [5]:
batting_2025 = pb.batting_stats_bref(2025)
show(batting_2025, n=3)


shape=(675, 28)  columns=28


,Name,Age,#days,Lev,Tm,G,PA,AB,R,H,2B,3B,HR,RBI,BB,IBB,SO,HBP,SH,SF,GDP,SB,CS,BA,OBP,SLG,OPS,mlbID
1,CJ Abrams,24,226,Maj-NL,Washington,144,635,580,92,149,35,5,19,60,37,1,125,14,1,3,5,31,3,0.257,0.315,0.433,0.748,682928
2,Wilyer Abreu,26,222,Maj-AL,Boston,116,422,378,53,92,17,0,22,69,40,5,104,0,1,3,8,6,3,0.243,0.314,0.463,0.777,677800
3,Maximo Acosta,22,240,Maj-NL,Miami,19,61,54,7,11,1,0,3,5,6,0,17,1,0,0,0,1,2,0.204,0.295,0.389,0.684,691185


In [6]:
# Hitters with a meaningful workload, sorted by OPS. OPS is a quick stand-in
# until we compute Barrel% / PullAir% / pitch run values per hitter from Statcast.
#
# Baseball-Reference uses `BA` (not `AVG`); other column names are mostly
# preserved as-is. We do a `[c for c in cols if c in df.columns]` filter so a
# silently-renamed column surfaces as a warning instead of crashing the cell.

regulars = batting_2025[batting_2025["PA"] >= 200].copy()

cols = ["Name", "Tm", "Age", "PA", "AB", "BA", "OBP", "SLG", "OPS", "OPS+", "HR", "BB", "SO"]
present = [c for c in cols if c in regulars.columns]
missing = [c for c in cols if c not in regulars.columns]
if missing:
    print(f"NOTE: columns not in batting_stats_bref output: {missing}")
    print(f"available columns: {list(regulars.columns)}")

sort_col = "OPS" if "OPS" in regulars.columns else present[-1]
regulars[present].sort_values(sort_col, ascending=False).head(15)


NOTE: columns not in batting_stats_bref output: ['OPS+']
available columns: ['Name', 'Age', '#days', 'Lev', 'Tm', 'G', 'PA', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'BB', 'IBB', 'SO', 'HBP', 'SH', 'SF', 'GDP', 'SB', 'CS', 'BA', 'OBP', 'SLG', 'OPS', 'mlbID']


,Name,Tm,Age,PA,AB,BA,OBP,SLG,OPS,HR,BB,SO
303,Aaron Judge,New York,33,710,567,0.339,0.462,0.688,1.150,54,128,165
455,Shohei Ohtani,Los Angeles,30,811,679,0.280,0.393,0.629,1.022,63,125,210
331,Nick Kurtz,Athletics,22,489,420,0.290,0.383,0.619,1.002,36,63,151
492,Cal Raleigh,Seattle,28,759,642,0.251,0.362,0.595,0.957,65,105,205
584,George Springer,Toronto,35,661,565,0.306,0.393,0.559,0.952,36,74,128
5,Ronald Acu\xc3\xb1a Jr.,Atlanta,27,412,338,0.290,0.417,0.518,0.935,21,71,102
554,Kyle Schwarber,Philadelphia,32,742,620,0.239,0.363,0.565,0.927,58,110,205
583,Juan Soto,New York,26,715,577,0.263,0.396,0.525,0.921,43,127,137
595,Kyle Stowers,Miami,27,457,399,0.288,0.368,0.544,0.912,25,48,125
586,Giancarlo Stanton,New York,35,311,275,0.265,0.342,0.564,0.906,24,32,102


---

## 4. Statcast for a specific pitcher

Pitch-level data — one row per pitch thrown. Useful columns for feature engineering:

| Column | What it captures |
|---|---|
| `release_speed` | Velocity. Trends down with fatigue / injury. |
| `release_spin_rate` | Spin. Predictive of pitch nastiness. |
| `pfx_x`, `pfx_z` | Horizontal / vertical movement in feet. |
| `pitch_type` | FF/SL/CH/CU etc. — pitch mix is a stylistic feature. |
| `estimated_woba_using_speedangle` | "Deserved" wOBA on contact. Stabilizes faster than actual wOBA. |
| `launch_speed`, `launch_angle` | When batted into play. |
| `delta_run_exp` | Change in run expectancy on this pitch (foundation of pitch run values). |
| `bb_type` | Statcast batted-ball type: `ground_ball`, `line_drive`, `fly_ball`, `popup`. |

Takes MLBAM ID, not name. Use `playerid_lookup` first.


In [7]:
skubal_id = int(pb.playerid_lookup("Skubal", "Tarik").iloc[0]["key_mlbam"])
skubal_pitches = pb.statcast_pitcher("2025-09-01", "2025-09-30", skubal_id)

print(f"pitches: {len(skubal_pitches)}")
skubal_pitches[["game_date", "pitch_type", "release_speed", "release_spin_rate", "pfx_x", "pfx_z", "estimated_woba_using_speedangle", "delta_run_exp"]].head(10)


pitches: 439


,game_date,pitch_type,release_speed,release_spin_rate,pfx_x,pfx_z,estimated_woba_using_speedangle,delta_run_exp
0,2025-09-30,SI,97.6,2333,1.22,1.19,0.187000,-0.353
1,2025-09-30,SI,98.3,2387,1.31,1.12,NaN,0.035
2,2025-09-30,CH,88.9,1915,1.64,0.07,0.691497,0.339
3,2025-09-30,SL,89.5,2264,-0.27,0.39,NaN,-0.087
4,2025-09-30,FF,96.8,2429,0.48,1.44,NaN,-0.088
5,2025-09-30,FF,98.0,2471,0.34,1.52,NaN,0.143
6,2025-09-30,SL,89.4,2184,-0.18,0.44,NaN,0.071
7,2025-09-30,CH,88.3,1828,1.34,0.33,NaN,0.039
8,2025-09-30,CH,91.5,1878,1.52,0.26,0.000000,-0.165
9,2025-09-30,CH,88.3,1786,1.37,0.14,NaN,-0.055


In [8]:
# Pitch run value per 100 pitches, by pitch type. This is the FG `wFB/100, wSL/100, ...`
# equivalent. Negative is good for the pitcher (he saved runs).
rv_per_100 = (
    skubal_pitches.groupby("pitch_type")["delta_run_exp"]
    .agg(["sum", "count"])
    .assign(rv_per_100=lambda d: -100.0 * d["sum"] / d["count"])  # flip sign: from pitcher's POV
    .sort_values("rv_per_100", ascending=False)
)
rv_per_100


,sum,count,rv_per_100
pitch_type,,,
CH,-2.526,117,2.158974
SL,-1.370,70,1.957143
FF,-2.012,133,1.512782
SI,-1.357,98,1.384694
CU,-0.170,21,0.809524


---

## 5. Statcast for a specific batter

Same Statcast table, filtered to a hitter's plate appearances. Computable from this view:

| Feature | How |
|---|---|
| `xwOBA` over window | Mean of `estimated_woba_using_speedangle` on PA-ending pitches |
| `Barrel%` | % of batted balls where `launch_speed_angle == 6` (Statcast's barrel bucket) |
| `HardHit%` | % of batted balls with `launch_speed >= 95` |
| `PullAir%` | % of `bb_type in {line_drive, fly_ball}` AND `hc_x` on pull side |
| Run value vs pitch type | `delta_run_exp` summed by `pitch_type`, then per 100 pitches seen |

The actual aggregation is your call (window length, sample-size floor).


In [9]:
judge_id = int(pb.playerid_lookup("Judge", "Aaron").iloc[0]["key_mlbam"])
judge_pitches = pb.statcast_batter("2025-09-01", "2025-09-30", judge_id)

batted = judge_pitches[judge_pitches["launch_speed"].notna()][
    ["game_date", "events", "launch_speed", "launch_angle", "estimated_woba_using_speedangle", "bb_type", "launch_speed_angle"]
]
print(f"batted-ball events: {len(batted)}")
batted.head(10)


batted-ball events: 111


,game_date,events,launch_speed,launch_angle,estimated_woba_using_speedangle,bb_type,launch_speed_angle
1,2025-09-30,NaN,80.7,14.0,NaN,NaN,NaN
2,2025-09-30,NaN,101.1,9.0,NaN,NaN,NaN
7,2025-09-30,field_out,90.3,41.0,0.0290,fly_ball,3.0
12,2025-09-30,single,116.1,5.0,0.6530,ground_ball,5.0
13,2025-09-30,single,100.4,5.0,0.6000,ground_ball,4.0
14,2025-09-28,single,101.7,5.0,0.6120,ground_ball,4.0
16,2025-09-28,field_out,103.5,55.0,0.0300,fly_ball,3.0
19,2025-09-28,field_out,89.8,8.0,0.4740,ground_ball,4.0
25,2025-09-28,field_out,73.1,-50.0,0.2030,ground_ball,2.0
29,2025-09-27,field_out,112.0,20.0,1.5776,line_drive,6.0


---

## 6. Team-level data

For matchup composites you'll roll up to team / lineup level eventually. The pybaseball ↔ Baseball-Reference team-level functions (`team_batting_bref`, `team_pitching_bref`, `schedule_and_record`) are currently broken — BR has changed page structure since pybaseball 2.2.7, and `schedule_and_record` additionally tripped over a pandas 3.x bug. So we use:

- `standings(year)` — still works; returns one DataFrame per division with W/L, games behind, etc. Useful as a label substrate for a naive baseline model.
- **MLB-StatsAPI** (Section 9 below) — the proper source for schedules, scores, probable pitchers, and lineups. Reliable JSON API, not a scrape.


In [10]:
# Three team-level queries that are still working:
#
#   - standings(year)                 -> W/L per team (list of DataFrames, one per division).
#                                        Used as label substrate for a naive baseline model.
#   - team_batting_bref(team, y0, y1) -> per-season batting totals for ONE team.
#                                        Note: first arg is a team abbrev, not a year.
#   - schedule_and_record(year, team) -> per-game results for ONE team
#                                        (the model's training labels).
#
# We're intentionally NOT pulling a cross-team batting/pitching leaderboard.
# The matchup engine builds team-level offense/defense by aggregating
# individual hitter and pitcher profiles weighted by lineup spot and
# expected IP — so the cross-team rollup is computed downstream, not fetched.
# (pybaseball's cross-team functions `team_batting`/`team_pitching` hit FG
# and are blocked.)

div_standings = pb.standings(2025)
print(f"divisions returned: {len(div_standings)}")
print("AL East:")
show(div_standings[0], n=5)

# Flatten to one DataFrame across all divisions, keep a division_idx column
# so you can groupby league/division later.
import pandas as _pd
standings_2025 = _pd.concat(
    div_standings, keys=range(len(div_standings)), names=["division_idx"]
).reset_index(level=0)
print(f"\nflattened standings shape: {standings_2025.shape}")
show(standings_2025, n=5)


divisions returned: 6
AL East:
shape=(5, 5)  columns=5

flattened standings shape: (30, 6)
shape=(30, 6)  columns=6


,division_idx,Tm,W,L,W-L%,GB
1,0,Toronto Blue Jays,94,68,.580,--
2,0,New York Yankees,94,68,.580,--
3,0,Boston Red Sox,89,73,.549,5.0
4,0,Tampa Bay Rays,77,85,.475,17.0
5,0,Baltimore Orioles,75,87,.463,19.0


---

## 7. Date-range stats (Baseball-Reference)

For rolling-window features ("how has this pitcher looked in the last 30 days?"):

- `pitching_stats_range(start_dt, end_dt)`
- `batting_stats_range(start_dt, end_dt)`

Slower than the season-level call (BR is scrape-heavy), but the only way to get true point-in-time aggregates from BR. Dates inclusive, `YYYY-MM-DD`.


In [11]:
last30_pitching = pb.pitching_stats_range("2025-09-01", "2025-09-30")
show(last30_pitching, n=5)


shape=(552, 41)  columns=41


,Name,Age,#days,Lev,Tm,G,GS,W,L,SV,IP,H,R,ER,BB,SO,HR,HBP,ERA,AB,2B,3B,IBB,GDP,SF,SB,CS,PO,BF,Pit,Str,StL,StS,GB/FB,LD,PU,WHIP,BAbip,SO9,SO/W,mlbID
1,Andrew Abbott,26,227,Maj-NL,Cincinnati,5,5,2.0,2.0,NaN,27.1,32,12,12,4,24,4,1,3.95,111,5,0,0,2,2,2,0,0,118,467,0.67,0.17,0.11,0.37,0.21,0.12,1.317,0.329,7.9,6.00,671096
2,Mick Abel,23,227,Maj-AL,Minnesota,2,1,1.0,NaN,NaN,10.0,4,2,2,4,15,0,0,1.80,33,0,0,0,1,0,0,0,0,37,144,0.64,0.15,0.15,0.50,0.11,0.06,0.800,0.222,13.5,3.75,690953
3,Philip Abner,23,226,Maj-NL,Arizona,5,0,NaN,NaN,NaN,3.2,5,2,2,3,5,0,0,4.91,15,2,0,0,1,0,0,0,0,18,71,0.63,0.17,0.11,0.30,0.10,0.30,2.182,0.500,12.3,1.67,691769
4,Bryan Abreu,28,228,Maj-AL,Houston,9,0,NaN,NaN,3.0,9.0,12,5,5,4,10,1,0,5.00,37,1,0,0,1,0,1,1,0,41,165,0.62,0.15,0.13,0.44,0.26,0.07,1.778,0.423,10.0,2.50,650556
5,Garrett Acton,27,240,Maj-AL,Tampa Bay,1,0,NaN,NaN,NaN,1.0,0,0,0,2,0,0,0,0.00,3,0,0,0,0,0,0,0,0,5,21,0.48,0.14,0.05,0.00,0.00,0.67,2.000,0.000,0.0,0.00,670183


---

## 8. Computed sabermetrics (`src/features/sabermetrics.py`)

Now we compute the FG-equivalents from the BR raw counts. `fip` / `xfip` / `siera` all vectorize across a DataFrame.

The constants used here (`fip_constant=3.13`, `lg_hr_per_fb=0.11`) are league averages; refine per-season once you have a clean league sum.


In [12]:
from src.features.sabermetrics import fip, xfip, siera, pythagorean_win_prob, american_odds_to_implied_prob, expected_value

# pitching_stats_bref gives us SO, BB, HBP, HR, IP, ERA, WHIP but NOT FIP/xFIP/SIERA.
# We compute FIP here. xFIP/SIERA additionally need raw FB/GB/LD/PU counts which
# BR's season-level table doesn't break out cleanly (it gives `GB/FB` as a ratio
# and `LD`/`PU` as counts, but not raw GB or FB) — we'll source those from
# Statcast aggregates in a later step and revisit this cell.

pit = pitching_2025[pitching_2025["GS"] >= 10].copy()
pit["FIP_calc"] = fip(pit["SO"], pit["BB"], pit["HBP"], pit["HR"], pit["IP"])

pit[["Name", "Tm", "IP", "ERA", "WHIP", "SO", "BB", "HR", "FIP_calc"]].sort_values("FIP_calc").head(15)


,Name,Tm,IP,ERA,WHIP,SO,BB,HR,FIP_calc
586,Shohei Ohtani,Los Angeles,67.1,3.34,1.069,90,16,5,2.176200
748,Tarik Skubal,Detroit,216.0,2.17,0.870,277,37,20,2.352222
747,Paul Skenes,Pittsburgh,187.2,1.97,0.948,216,42,11,2.355427
647,Cole Ragans,Kansas City,61.2,4.67,1.184,98,20,7,2.492745
705,Cristopher S\xc3\xa1nchez,Philadelphia,214.0,2.48,1.061,225,47,12,2.527196
857,Logan Webb,San Francisco,207.0,3.22,1.237,224,46,14,2.598599
703,Chris Sale,Atlanta,125.2,2.58,1.066,165,32,11,2.666741
219,Nathan Eovaldi,Texas,130.0,1.73,0.854,129,21,10,2.791538
684,Trevor Rogers,Baltimore,109.2,1.81,0.903,103,29,6,2.809487
473,Jes\xc3\xbas Luzardo,Philadelphia,191.1,3.86,1.202,224,58,16,2.831727


In [13]:
# Quick demo of the pythagorean -> EV chain on hypothetical projected runs.
# Replace these with real numbers once the matchup engine is implemented.
home_runs, away_runs = 4.8, 4.1
p_home = pythagorean_win_prob(home_runs, away_runs)

# Suppose the book is offering the home team at -135 (favorite).
home_odds = -135
implied = american_odds_to_implied_prob(home_odds)
ev = expected_value(p_home, home_odds, stake=1.0)

print(f"projected: home={home_runs}, away={away_runs}")
print(f"model P(home wins) = {p_home:.4f}")
print(f"book implied  P    = {implied:.4f}  (at {home_odds})")
print(f"EV per $1 staked   = ${ev:.4f}  ({'BET' if ev > 0 else 'SKIP'})")


projected: home=4.8, away=4.1
model P(home wins) = 0.5716
book implied  P    = 0.5745  (at -135)
EV per $1 staked   = $-0.0050  (SKIP)


---

### 8b. Computing SIERA from Statcast

`pitching_stats_bref` doesn't give us SIERA and FanGraphs is blocked, so we compute it ourselves. SIERA needs per-pitcher counts of SO, BB, GB, FB, popups, and PA — all derivable from Statcast pitch-level data.

`src/features/statcast_aggregation.py` does the reshape:

| Helper | What it does |
|---|---|
| `pa_end_pitches(df)` | Filter Statcast pitches to the last pitch of each PA (one row per PA). |
| `aggregate_pa_counts(df, by='pitcher')` | Groupby pitcher (or batter) and count SO / BB / HBP / HR / GB / FB / PU / LD / PA / BIP. |
| `add_rate_stats(agg)` | Append K%, BB%, HR%, GB%, FB%, LD%, PU% rate columns. |
| `attach_player_names(agg)` | Join MLBAM IDs to readable names via `pb.playerid_reverse_lookup`. |

Then `siera(SO, BB, GB, FB, PU, PA)` from `sabermetrics.py` finishes the job.

> Note: `pb.statcast(start, end)` for a full month is ~700K rows and takes ~30–60s on first call. Subsequent runs hit the local cache. Adjust the date window to balance freshness vs. wait.

In [ ]:
from src.features.statcast_aggregation import aggregate_pa_counts, add_rate_stats, attach_player_names
from src.features.sabermetrics import siera

# 30-day window. Shorter = faster but fewer qualified pitchers.
SC_START, SC_END = "2025-09-01", "2025-09-30"
sc = pb.statcast(SC_START, SC_END, verbose=False)
print(f"statcast rows: {len(sc):,}  unique pitchers: {sc['pitcher'].nunique()}")

# Aggregate -> rates -> SIERA -> names
pitcher_agg = aggregate_pa_counts(sc, by="pitcher")
pitcher_agg = add_rate_stats(pitcher_agg)
pitcher_agg["SIERA"] = siera(
    pitcher_agg["SO"], pitcher_agg["BB"],
    pitcher_agg["GB"], pitcher_agg["FB"], pitcher_agg["PU"],
    pitcher_agg["PA"],
)
pitcher_agg = attach_player_names(pitcher_agg, id_col="pitcher")

# Sample-size floor — SIERA stabilizes around 150+ PA, but for a 30-day window
# we relax to 80+ so the leaderboard isn't empty.
qualified = pitcher_agg[pitcher_agg["PA"] >= 80].copy()
print(f"qualified pitchers (PA>=80): {len(qualified)}")

qualified[
    ["player_name", "PA", "BIP", "SO", "BB", "HR", "GB", "FB", "LD", "PU",
     "K_pct", "BB_pct", "GB_pct", "SIERA"]
].sort_values("SIERA").head(15)

---

## 9. MLB-StatsAPI (schedules, probable pitchers, lineups)

`MLB-StatsAPI` is a thin Python wrapper around MLB's official stats endpoint. Unlike pybaseball it's not scraping — it's a documented JSON API and is reliable.

Key calls:

| Call | What it returns |
|---|---|
| `statsapi.schedule(date='MM/DD/YYYY')` | List of games with `game_id`, `home_name`, `away_name`, `home_probable_pitcher`, `away_probable_pitcher`, `venue`, `home_score`, `away_score`, `status` |
| `statsapi.boxscore_data(gamePk)` | Full box score: lineups, pitchers, results |
| `statsapi.lookup_player('Aaron Judge')` | Name -> player IDs |
| `statsapi.get('schedule', {'date': ..., 'hydrate': 'probablePitcher,lineups,weather,venue'})` | Lower-level call with hydration |

For the matchup engine we need **probables** (always available a day ahead) and **lineups** (typically 2-4 hours pre-game).


In [15]:
import statsapi
from datetime import date

today = date.today().strftime("%m/%d/%Y")
games = statsapi.schedule(date=today)
print(f"{len(games)} games today")

# Convenient frame view
games_df = pd.DataFrame(games)[
    ["game_id", "game_datetime", "away_name", "home_name", "venue_name",
     "away_probable_pitcher", "home_probable_pitcher", "status"]
]
games_df


15 games today


,game_id,game_datetime,away_name,home_name,venue_name,away_probable_pitcher,home_probable_pitcher,status
0,824841,2026-05-13T17:05:00Z,New York Yankees,Baltimore Orioles,Oriole Park at Camden Yards,Max Fried,Kyle Bradish,Scheduled
1,824438,2026-05-13T17:10:00Z,Los Angeles Angels,Cleveland Guardians,Progressive Field,Reid Detmers,Parker Messick,Scheduled
2,824519,2026-05-13T22:40:00Z,Washington Nationals,Cincinnati Reds,Great American Ball Park,Jake Irvin,Nick Lodolo,Scheduled
3,823386,2026-05-13T22:40:00Z,Colorado Rockies,Pittsburgh Pirates,PNC Park,Jose Quintana,Mitch Keller,Scheduled
4,824763,2026-05-13T22:45:00Z,Philadelphia Phillies,Boston Red Sox,Fenway Park,Andrew Painter,Sonny Gray,Scheduled
5,822815,2026-05-13T23:07:00Z,Tampa Bay Rays,Toronto Blue Jays,Rogers Centre,Griffin Jax,Dylan Cease,Scheduled
6,823631,2026-05-13T23:10:00Z,Detroit Tigers,New York Mets,Citi Field,Framber Valdez,Christian Scott,Scheduled
7,824928,2026-05-13T23:15:00Z,Chicago Cubs,Atlanta Braves,Truist Park,Shota Imanaga,JR Ritchie,Scheduled
8,824603,2026-05-13T23:40:00Z,Kansas City Royals,Chicago White Sox,Rate Field,Seth Lugo,Noah Schultz,Scheduled
9,823710,2026-05-13T23:40:00Z,Miami Marlins,Minnesota Twins,Target Field,Max Meyer,Simeon Woods Richardson,Scheduled


In [16]:
# Drill into one game for lineup + weather hydration. Lineups are populated
# 2-4 hours before first pitch; before that, the `lineups` field is empty.
if not games_df.empty:
    sample_game_id = int(games_df.iloc[0]["game_id"])
    hydrated = statsapi.get(
        "schedule",
        {"date": today, "sportId": 1, "hydrate": "probablePitcher,lineups,weather,venue,team"},
    )
    # Surface lineups for the sample game
    for date_block in hydrated.get("dates", []):
        for g in date_block.get("games", []):
            if g["gamePk"] == sample_game_id:
                print("home probable:", g["teams"]["home"].get("probablePitcher", {}).get("fullName"))
                print("away probable:", g["teams"]["away"].get("probablePitcher", {}).get("fullName"))
                print("lineups available:", "lineups" in g)
                break


home probable: Kyle Bradish
away probable: Max Fried
lineups available: True


---

## 10. Saving DataFrames for later

When a pull looks like a feature candidate, persist it. `save_raw` writes Parquet to `data/raw/pybaseball/` with a UTC timestamp suffix so successive saves don't overwrite. `load_raw` resolves either an exact filename or the most-recent match for a prefix.


In [17]:
save_raw(pitching_2025, "pitching_stats_bref_2025")
save_raw(batting_2025,  "batting_stats_bref_2025")
save_raw(standings_2025, "standings_2025")
save_raw(pitcher_agg, f"statcast_pitcher_agg_{SC_START}_to_{SC_END}")
# games_df is the MLB-StatsAPI schedule for today (from Section 9)
save_raw(games_df, f"statsapi_schedule_{date.today().isoformat()}")

list_raw()


[PosixPath('/Users/danielmatten/mlb-ev/data/raw/pybaseball/batting_stats_bref_2025_2026-05-13T04-56-01Z.parquet'),
 PosixPath('/Users/danielmatten/mlb-ev/data/raw/pybaseball/batting_stats_bref_2025_2026-05-13T05-00-58Z.parquet'),
 PosixPath('/Users/danielmatten/mlb-ev/data/raw/pybaseball/batting_stats_bref_2025_2026-05-13T05-07-35Z.parquet'),
 PosixPath('/Users/danielmatten/mlb-ev/data/raw/pybaseball/batting_stats_bref_2025_2026-05-13T05-14-15Z.parquet'),
 PosixPath('/Users/danielmatten/mlb-ev/data/raw/pybaseball/pitching_stats_bref_2025_2026-05-13T04-56-01Z.parquet'),
 PosixPath('/Users/danielmatten/mlb-ev/data/raw/pybaseball/pitching_stats_bref_2025_2026-05-13T05-00-58Z.parquet'),
 PosixPath('/Users/danielmatten/mlb-ev/data/raw/pybaseball/pitching_stats_bref_2025_2026-05-13T05-07-35Z.parquet'),
 PosixPath('/Users/danielmatten/mlb-ev/data/raw/pybaseball/pitching_stats_bref_2025_2026-05-13T05-14-15Z.parquet'),
 PosixPath('/Users/danielmatten/mlb-ev/data/raw/pybaseball/standings_2025_20

In [18]:
pit_reloaded = load_raw("pitching_stats_bref_2025")
pit_reloaded.shape


(873, 41)

---

## 11. Matchup engine: decisions locked in

The next step beyond exploration is the matchup engine in `src/features/matchup.py`. The runtime flow:

```
for each game today (MLB-StatsAPI):
    home_lineup, away_lineup  = lookup_or_project_lineup(game)   # see decision #5
    home_starter, away_starter = lookup_probable_pitchers(game)
    home_bullpen, away_bullpen = recent_bullpen(team)

    # Feature vectors from saved pulls in data/raw/pybaseball/
    profiles_home_hitters  = lookup_hitter_profile (home_lineup, as_of=snapshot_date)
    profiles_away_pitching = lookup_pitcher_profile(away_starter + away_bullpen, ...)
    ...

    home_runs = project_runs(profiles_home_hitters, profiles_away_pitching, defense_away, context)
    away_runs = project_runs(profiles_away_hitters, profiles_home_pitching, defense_home, context)

    p_home = pythagorean_win_prob(home_runs, away_runs)

    # From the odds snapshot in S3
    home_odds, away_odds = best_price_from_books(game, snapshot)
    ev_home = expected_value(p_home,        home_odds)
    ev_away = expected_value(1 - p_home,    away_odds)
```

### Locked-in design parameters

| # | Decision | Choice |
|---|---|---|
| 1 | Lineup weighting | `LINEUP_PA_WEIGHTS` — PA share by batting order, **normalized to 1.0** (pinch hitters / bench bats excluded). Leadoff ~13.1%, #9 ~9.4%. |
| 2 | SP / bullpen blend | starter weight = `clamp(expected_ip_per_start / 9, 0.25, 0.85)`; bullpen takes the remainder, equal-weighted. |
| 3 | Defense (OAA) | **Scalar multiplier** on contact-allowed runs, calibrated so +25 OAA ≈ -2.75% RA. v2: replace with learned interaction term once MVP is calibrated. |
| 4 | Park / weather | **Weather ignored.** Park factor applied to the **air-ball share** of runs only (see recipe below). |
| 5 | Projected lineup | Last-7-day modal lineup, **platoon-split** by opposing-starter handedness. Implemented in `src/features/lineup_projection.py` (stub). |

### Decision #4 recipe (park-air-only adjustment)

```python
# 1. Blend pitcher and hitter contact rates (60/40, pitcher-favored).
gb, fb, ld = blend_contact_profile(pitcher_gb, pitcher_fb, pitcher_ld,
                                   hitter_gb,  hitter_fb,  hitter_ld)

# 2. Convert to ground vs air run-share via league wOBA on contact
#    (GB .210, FB .340, LD .685).
ground_share, air_share = air_share_from_contact(gb, fb, ld)

# 3. Park factor multiplies the AIR share only.
final_runs = baseline_runs * (ground_share + air_share * park_run_factor)
```

For a neutral matchup (pitcher and hitter both league-average), this gives `ground_share ≈ 0.263`, `air_share ≈ 0.737`. Coors's 1.20 park factor becomes an effective ~1.147 multiplier on the matchup; Petco's 0.95 becomes ~0.963.

### What's still TBD

The only piece blocking `project_runs` end-to-end is the **baseline-runs combiner** (step 3 in the function): how do per-team composites of wRC+ / xwOBA / Barrel% / SIERA / xFIP turn into a neutral-context run total? That's a small regression once we have a few weeks of saved feature snapshots paired with actual game scores. Until then, every other primitive in `matchup.py` is implemented and unit-testable.


---

## 12. Game outcomes (labels for training)

`src.ingest.fetch_outcomes` mirrors the odds-ingestion pattern: hits MLB-StatsAPI, wraps the response with capture metadata, and writes one JSON per *game date* to either local disk or S3.

CLI cheat sheet:

```bash
# nightly (default = yesterday in UTC) — what the Lambda runs
python -m src.ingest.fetch_outcomes

# explicit single date
python -m src.ingest.fetch_outcomes --date 2025-08-14

# backfill a range (writes one file per day, ~0.5s/day on the API)
python -m src.ingest.fetch_outcomes --start 2024-03-20 --end 2025-11-05
```

`src.features.outcomes_loader` flattens the raw JSON into a tidy DataFrame and rolls each season up to `data/outcomes/outcomes_{year}.parquet`. Filters applied by default:

- `status` ∈ `{Final, Completed Early}` (drops postponed, suspended, in-progress)
- `game_type` ∈ `{R, F, D, L, W}` (regular season + wild card/div/LCS/WS; drops spring, all-star, exhibition)

This is the LABEL side of the training set. The next session wires the FEATURE side: for each game, build per-team feature vectors *as of the game date* (no lookahead).

In [ ]:
import pandas as pd
from src.features.outcomes_loader import load_outcomes

games = pd.concat([load_outcomes(2024), load_outcomes(2025)], ignore_index=True)

print(f"games:           {len(games):,}")
print(f"date range:      {games.game_date.min()} -> {games.game_date.max()}")
print(f"home win rate:   {games.home_win.mean():.3%}")
print(f"mean total runs: {games.total_runs.mean():.2f}")
print(f"mean home/away:  {games.home_score.mean():.2f} / {games.away_score.mean():.2f}")
print(f"postseason:      {games.is_postseason.sum()} games")
print()

games.head(8)[
    ["game_date", "away_name", "home_name", "away_score", "home_score",
     "home_win", "venue_name"]
]

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

games["run_diff"].hist(bins=range(-20, 21), edgecolor="black", ax=axes[0])
axes[0].set_xlabel("home_score - away_score")
axes[0].set_ylabel("games")
axes[0].set_title(f"Run diff (mean {games.run_diff.mean():.2f})")
axes[0].axvline(0, color="red", linestyle="--", alpha=0.5)

games["total_runs"].hist(bins=range(0, 30), edgecolor="black", ax=axes[1])
axes[1].set_xlabel("total runs")
axes[1].set_title(f"Total runs (mean {games.total_runs.mean():.2f})")

plt.tight_layout()
plt.show()

home_records = games.groupby("home_name")["home_win"].agg(["count", "mean"]).round(3)
home_records.columns = ["home_games", "home_win_pct"]
home_records.sort_values("home_win_pct", ascending=False).head(10)

---

## 13. Point-in-time feature aggregator

Training the matchup model needs, for every game, the features available **strictly before** first pitch. Re-pulling Statcast per game date would be insane; instead we pull each full season once into `data/raw/statcast/statcast_{year}.parquet` (see `src.ingest.fetch_statcast_history`) and build a cumulative-counts index keyed by `(player, date)`.

```bash
# one-time (or whenever a season ends and you want fresh data):
python -m src.ingest.fetch_statcast_history --years 2024 2025
```

`src.features.cumulative` then turns that into:

| Helper | What it does |
|---|---|
| `load_statcast(years)` | Read + concat season parquets. |
| `build_cumulative(df, group='pitcher')` | Per-(player, season) daily-bucketed running totals of PA, SO, BB, GB, FB, LD, BARREL, wOBA num/den. |
| `compute_rate_stats(cum)` | Append SIERA, K%, BB%, GB%/FB%/LD%/PU%, Barrel%, xwOBA. |
| `lookup_asof(cum, player_id, date)` | Single-player query — returns the latest row strictly before `date` (and strictly within the same season, by default). |
| `join_features_asof(games, cum, by=..., prefix=...)` | Bulk join via `pd.merge_asof(direction='backward', allow_exact_matches=False)`. The training-set workhorse. |

The bulk join uses `direction='backward'` + `allow_exact_matches=False` so the as-of row is always **strictly before** the game date — no lookahead, ever.

In [ ]:
from src.features.cumulative import (
    load_statcast, build_cumulative, compute_rate_stats,
    lookup_asof, join_features_asof,
)

sc = load_statcast([2024, 2025])
print(f"statcast rows: {len(sc):,}  pitchers: {sc.pitcher.nunique()}  batters: {sc.batter.nunique()}")

cum_p = compute_rate_stats(build_cumulative(sc, group="pitcher"))
cum_b = compute_rate_stats(build_cumulative(sc, group="batter"))
print(f"pitcher rows: {len(cum_p):,}   batter rows: {len(cum_b):,}")

# End-of-2024 SIERA leaderboard (qualifying = 600+ PA).
eoy = (
    cum_p[cum_p.game_year == 2024]
    .sort_values(["player_id", "game_date"])
    .groupby("player_id").tail(1)
)
eoy[eoy.PA_cum >= 600].nsmallest(15, "SIERA")[
    ["player_id", "PA_cum", "K_pct", "BB_pct", "GB_pct", "Barrel_pct", "xwOBA", "SIERA"]
].round(3)

In [ ]:
# Single-player as-of lookups. Hardcoded MLBAM IDs (faster than playerid_lookup).
KNOWN_PITCHERS = {"Skubal": 669373, "Crochet": 676979, "Sale": 519242,
                  "Wheeler": 554430, "Kirby": 669923}
KNOWN_HITTERS  = {"Judge": 592450, "Ohtani": 660271, "Soto": 665742,
                  "Witt Jr": 677951, "Riley": 663586}

print("End-of-2025 pitchers:")
for name, pid in KNOWN_PITCHERS.items():
    r = lookup_asof(cum_p, pid, "2026-01-01", same_season_only=False)
    if r is None: continue
    print(f"  {name:>8}: PA={int(r.PA_cum):>4}  K%={r.K_pct:.1%}  BB%={r.BB_pct:.1%}  "
          f"GB%={r.GB_pct:.1%}  Barrel%={r.Barrel_pct:.1%}  xwOBA={r.xwOBA:.3f}  SIERA={r.SIERA:.2f}")

print("\nEnd-of-2025 hitters:")
for name, pid in KNOWN_HITTERS.items():
    r = lookup_asof(cum_b, pid, "2026-01-01", same_season_only=False)
    if r is None: continue
    print(f"  {name:>8}: PA={int(r.PA_cum):>4}  K%={r.K_pct:.1%}  BB%={r.BB_pct:.1%}  "
          f"HR%={r.HR_pct:.1%}  Barrel%={r.Barrel_pct:.1%}  xwOBA={r.xwOBA:.3f}")

In [ ]:
# Bulk join: attach point-in-time pitcher features to a list of games.
# Each row gets the cumulative state of the pitcher STRICTLY BEFORE the game date.
demo_games = pd.DataFrame({
    "game_date": pd.to_datetime(["2024-08-01", "2025-08-15", "2025-09-15",
                                  "2025-07-04", "2025-05-01"]),
    "home_pitcher_id": [669373, 669373, 676979, 554430, 519242],
    "pitcher_name":    ["Skubal", "Skubal", "Crochet", "Wheeler", "Sale"],
})

joined = join_features_asof(
    demo_games, cum_p, by="home_pitcher_id", prefix="sp_",
    feature_cols=["SIERA", "K_pct", "BB_pct", "Barrel_pct", "xwOBA", "PA_cum"],
)
joined.round(3)

# Note how Skubal's SIERA descends across his two 2025 dates (point-in-time,
# not season-final) and resets between 2024 and 2025 because same_season_only=True.

---

## 14. Training feature vectors (per game)

This is where the four prior pieces converge into one row per game:

| Source | Module | What it contributes |
|---|---|---|
| Labels | `src.features.outcomes_loader` | `home_score`, `away_score`, `home_win`, `run_diff`, `total_runs` |
| Lineups & starters | `src.features.lineup_loader` | Who actually played: 9 batters per side + starter |
| Point-in-time stats | `src.features.cumulative` | Cumulative pitcher & batter aggregates as-of game date |
| Combiner | `src.features.build_features` | Joins all of the above into one wide row |

### Backfill workflow (already run for 2024 + 2025)

```bash
# 1. raw boxscores: one JSON per game in data/raw/boxscores/<year>/
python -m src.ingest.fetch_boxscores --years 2024 2025

# 2. flatten -> data/lineups/lineups_<year>.parquet (+ long form + name cache)
python -c "from src.features.lineup_loader import rollup_to_parquet; \
           [rollup_to_parquet(y) for y in (2024, 2025)]"

# 3. build training set -> data/features/training_<year>.parquet
python -c "from src.features.build_features import build_training_features; \
           [build_training_features(y) for y in (2024, 2025)]"
```

The starter side is straightforward: one `merge_asof` (`backward`, `allow_exact_matches=False`) per side gives us the starter's cumulative state strictly before first pitch. The hitter side fans out via `explode_lineups`, joins per (game, slot, player_id), then collapses back into one `<side>_off_<feature>` column per metric using `LINEUP_PA_WEIGHTS` — re-normalizing within each game when some hitters are below the 25-PA sample floor.

No park factor, no OAA, no rest-day adjustment yet — those are static team / venue lookups that we'll layer on once the baseline regressor has been fit and we know which residuals they help with.


In [ ]:
import pandas as pd
from src.features.lineup_loader import load_player_names

train = pd.concat(
    [pd.read_parquet(f"data/features/training_{y}.parquet") for y in (2024, 2025)],
    ignore_index=True,
)
names = load_player_names()

print(f"games:        {len(train):,}")
print(f"date range:   {train.game_date.min().date()} -> {train.game_date.max().date()}")
print(f"feature cols: {len(train.columns)}")
print()
print("Feature coverage:")
for c in ["home_sp_SIERA", "home_sp_xwOBA", "away_sp_SIERA", "away_sp_xwOBA",
          "home_off_xwOBA", "away_off_xwOBA", "home_off_Barrel_pct", "away_off_Barrel_pct"]:
    print(f"  {c:<28} {train[c].notna().mean():.1%} non-null")


In [ ]:
# Sanity-check one game end-to-end: Cubs @ Pirates, 2025-09-15.
row = train[train.game_id == 776318].iloc[0]
print(f"{row.away_name} @ {row.home_name}  ({row.game_date.date()})")
print(f"  final: {row.away_name} {int(row.away_score)}, {row.home_name} {int(row.home_score)}")
print()
print(f"  away starter: {names.get(int(row.away_starter_id), '?')} "
      f"(SIERA={row.away_sp_SIERA:.2f}, K%={row.away_sp_K_pct:.1%}, wOBA={row.away_sp_xwOBA:.3f}, PA={int(row.away_sp_PA_cum)})")
print(f"  home starter: {names.get(int(row.home_starter_id), '?')} "
      f"(SIERA={row.home_sp_SIERA:.2f}, K%={row.home_sp_K_pct:.1%}, wOBA={row.home_sp_xwOBA:.3f}, PA={int(row.home_sp_PA_cum)})")
print()
print(f"  away lineup wOBA composite: {row.away_off_xwOBA:.3f}  "
      f"barrel%: {row.away_off_Barrel_pct:.1%}  K%: {row.away_off_K_pct:.1%}")
print(f"  home lineup wOBA composite: {row.home_off_xwOBA:.3f}  "
      f"barrel%: {row.home_off_Barrel_pct:.1%}  K%: {row.home_off_K_pct:.1%}")
print()
print("  away lineup:", [names.get(int(p), p) for p in row.away_lineup])
print("  home lineup:", [names.get(int(p), p) for p in row.home_lineup])


In [ ]:
# Does the SIERA we computed for each game actually predict opposing-team scoring?
# This is the first-pass "are these features useful?" check.

import matplotlib.pyplot as plt

# Stratify by starter SIERA quintile -> avg runs allowed (= other team's score).
def stratify(df, feat, target, q=5):
    sub = df.dropna(subset=[feat, target]).copy()
    sub["bucket"] = pd.qcut(sub[feat], q, labels=False, duplicates="drop")
    g = sub.groupby("bucket").agg(
        bucket_min=(feat, "min"), bucket_max=(feat, "max"),
        n=(target, "size"), mean_target=(target, "mean"),
    ).round(3)
    return g

print("Home SP SIERA quintile -> avg AWAY-team runs scored")
print(stratify(train, "home_sp_SIERA", "away_score"))
print()
print("Away SP SIERA quintile -> avg HOME-team runs scored")
print(stratify(train, "away_sp_SIERA", "home_score"))
print()
print("Note the monotone trend: better SIERA (lower bucket index) holds opposing")
print("teams below ~4.1 runs/game; worst SIERA gives up ~4.7 — a 0.6-run swing.")
print("That's the matchup engine's primary signal showing up cleanly in raw form.")

# Scatter of run_diff vs the simplest matchup edge: starter SIERA differential.
train["run_diff"] = train["home_score"].astype(int) - train["away_score"].astype(int)
train["sp_siera_diff"] = train["away_sp_SIERA"] - train["home_sp_SIERA"]  # positive = home pitcher better
mask = train["sp_siera_diff"].notna() & train["run_diff"].notna()
plt.figure(figsize=(7, 4))
plt.scatter(train.loc[mask, "sp_siera_diff"], train.loc[mask, "run_diff"], alpha=0.15, s=8)
plt.axhline(0, color="red", linestyle="--", alpha=0.4)
plt.axvline(0, color="red", linestyle="--", alpha=0.4)
plt.xlabel("away_sp_SIERA - home_sp_SIERA  (positive = home SP better)")
plt.ylabel("home_score - away_score")
corr = train.loc[mask, ["sp_siera_diff", "run_diff"]].corr().iloc[0, 1]
plt.title(f"Run differential vs starter SIERA differential  (r={corr:+.3f}, n={mask.sum():,})")
plt.tight_layout()
plt.show()


---

## 15. Baseline runs model + Pythag win probability

The first end-to-end model. The matchup design has two halves:

1. **A regression that maps (team's offensive composite, opposing starter's profile, context) → expected runs.**
2. **The Pythagorean formula** that turns `(home_runs_pred, away_runs_pred)` into `p_home`.

We train ONE regression on side-stacked data: every game produces two training rows (home-as-offense + away-as-offense), with the relevant columns renamed side-agnostically (`off_xwOBA`, `opp_sp_SIERA`, etc.). At inference we call it twice per game.

### Architecture choices

| Choice | Why |
|---|---|
| Ridge regression | Interpretable, stable when features correlate (SIERA / K% / xwOBA all move together), fast to retrain. |
| Side stacking | Doubles the training data and gives a clean home-field-advantage coefficient via the ``is_home`` feature. |
| ``feature_cols`` as a parameter | Adding Option-B features (park factor, OAA, rolling-window stats, rest days) is a one-line change. No surgery on the model code. |
| Train 2022-2024, test 2025 | Out-of-sample, time-respecting. The final cell of this section shows an ablation across training-window choices — adding 2022 and 2023 **monotonically reduces** discriminative power because of the 2023 pitch-clock regime change. Recent + features beats more data + stale features. |
| Pythag exponent 1.83 | Bill James's classic value. We can tune later. |
| Slim feature set | The kitchen-sink set (SIERA + its K%/BB%/GB%/FB% components together) gives Ridge enough collinearity to flip the SIERA coefficient negative. The slim set keeps one summary stat per "tier" — coefficients then come out signed the way baseball intuition demands. ``KITCHEN_SINK_FEATURE_COLS`` is kept for ablation. |

### What good looks like

The right metrics to watch are **calibration** and **bucketed accuracy**, not raw accuracy alone. Raw accuracy is bounded by "always pick home" (≈54%) because individual baseball games are so noisy. What matters for betting is whether the model's confident picks are MORE accurate than its low-confidence ones — i.e., does the calibration curve climb monotonically?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.model.runs_model import (
    train_runs_model, predict_runs, coefficient_table,
    DEFAULT_FEATURE_COLS, KITCHEN_SINK_FEATURE_COLS,
)
from src.model.evaluate import (
    runs_metrics, pythag_win_prob, pythag_metrics, calibration_table,
)

TRAIN_YEARS = (2022, 2023, 2024)
TEST_YEAR   = 2025

train = pd.concat(
    [pd.read_parquet(f"data/features/training_{y}.parquet") for y in TRAIN_YEARS],
    ignore_index=True,
)
test = pd.read_parquet(f"data/features/training_{TEST_YEAR}.parquet")
train["home_win"] = (train["home_score"] > train["away_score"]).astype(int)
test["home_win"]  = (test["home_score"]  > test["away_score"]).astype(int)

print(f"train {list(TRAIN_YEARS)}: {len(train):,} games | "
      f"home win rate {train.home_win.mean():.3%} | "
      f"avg runs/team {train[['home_score','away_score']].stack().mean():.2f}")
print(f"test  {TEST_YEAR}:        {len(test):,} games | "
      f"home win rate {test.home_win.mean():.3%}  | "
      f"avg runs/team {test[['home_score','away_score']].stack().mean():.2f}")
print()

# Home/away asymmetry by year — the is_home learning signal.
print("Home-field scoring advantage by year (training signal for is_home):")
for y in (*TRAIN_YEARS, TEST_YEAR):
    df = pd.read_parquet(f"data/features/training_{y}.parquet")
    h, a = df.home_score.mean(), df.away_score.mean()
    wr = (df.home_score > df.away_score).mean()
    print(f"  {y}: home={h:.3f}  away={a:.3f}  diff={h-a:+.3f}  home_win={wr:.3%}")
print()
print("Notice 2022 has +0.07 runs home advantage but 2023-2024 have NEGATIVE.")
print("2025 has +0.09 — so the model's is_home coef will be small and noisy.")
print()

print(f"feature cols ({len(DEFAULT_FEATURE_COLS)} = slim set):")
for c in DEFAULT_FEATURE_COLS:
    print(f"  - {c}")

model = train_runs_model(train)
print(f"\nRidge fit on {model.train_n:,} stacked rows (alpha={model.model.alpha}).")
print(f"Target mean runs: {model.target_mean:.3f}   std: {model.target_std:.3f}")


In [ ]:
# Coefficients (standardized scale -> each row says "+1 sigma of this
# feature shifts predicted runs by COEF").
print("=== Coefficients with the slim feature set ===")
print(coefficient_table(model).to_string(index=False))

# Ablation: same data, but with the kitchen-sink feature set. Notice SIERA's
# coefficient flips sign because K%/BB% absorb most of the SIERA-shaped variance,
# leaving SIERA with the residual — which Ridge happens to fit negative.
ks_model = train_runs_model(train, feature_cols=KITCHEN_SINK_FEATURE_COLS)
print()
print("=== Same data, KITCHEN_SINK feature set (collinearity demo) ===")
print(coefficient_table(ks_model).to_string(index=False))
print()
print("Reading: with the slim set, opp_sp_SIERA is the dominant POSITIVE coefficient")
print("(higher opponent SIERA -> we score more, exactly right). With the kitchen-sink")
print("set it FLIPS NEGATIVE because K%/BB% are mathematically inside SIERA and the")
print("solver redistributes the weight arbitrarily. This is why DEFAULT_FEATURE_COLS")
print("only keeps one summary stat per tier — coefficients have to be interpretable")
print("when we're using them to reason about bet rationale.")


In [ ]:
pred = predict_runs(model, test)
p_home = pythag_win_prob(pred["home_runs_pred"], pred["away_runs_pred"])

print("=== Per-side runs regression on TEST (2025) ===")
rh = runs_metrics(pred["home_score"], pred["home_runs_pred"])
ra = runs_metrics(pred["away_score"], pred["away_runs_pred"])
print(f"home runs:  MAE={rh.mae:.3f}  RMSE={rh.rmse:.3f}  R^2={rh.r2:+.4f}  bias={rh.bias:+.3f}")
print(f"away runs:  MAE={ra.mae:.3f}  RMSE={ra.rmse:.3f}  R^2={ra.r2:+.4f}  bias={ra.bias:+.3f}")
print()
print("R^2 < 0.01 looks alarming but is expected: a single game scores a 38-PA")
print("sample. The point is whether the SHIFT in predicted runs tracks the SHIFT")
print("in actual runs across many games — which calibration below addresses.")
print()

# Model vs trivial baselines on win prediction
m = pythag_metrics(test["home_win"], p_home)
y = test["home_win"].to_numpy()
acc_always_home = (y == 1).mean()
p_const = np.full(len(test), m.home_win_rate)
m_const = pythag_metrics(y, p_const)
mask = test["home_sp_SIERA"].notna() & test["away_sp_SIERA"].notna()
acc_siera = ((test.loc[mask, "home_sp_SIERA"] < test.loc[mask, "away_sp_SIERA"]).astype(int).to_numpy()
             == y[mask.to_numpy()]).mean()

print("=== Win prediction on TEST (2025) ===")
print(f"  Ridge + Pythag   :  acc={m.accuracy:.3%}  log_loss={m.log_loss:.4f}  Brier={m.brier:.4f}")
print(f"                       mean p_home={m.mean_p_home:.3f}  (actual {m.home_win_rate:.3%})")
print()
print("Trivial baselines:")
print(f"  B1 always home   :  acc={acc_always_home:.3%}")
print(f"  B2 constant p={m.home_win_rate:.3f}:  log_loss={m_const.log_loss:.4f}  Brier={m_const.brier:.4f}")
print(f"  B3 lower SIERA   :  acc={acc_siera:.3%}  (n={mask.sum()})")
print()
print("Reading: the model's raw accuracy doesn't beat 'always home' because the home")
print("base rate (54%) is itself a strong predictor on noisy single games. The story")
print("the model has to tell is *where* the wins are concentrated — see calibration.")


In [ ]:
# Calibration: bucket games by model-predicted p_home, then see what
# fraction actually went home. If our predictions are well-calibrated AND
# discriminative, the bars climb monotonically from left to right.
calib = calibration_table(p_home, test["home_win"], bins=10)
print("=== Calibration (10 quantile buckets, lowest p_home -> highest) ===")
print(calib.to_string())

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot([0.3, 0.7], [0.3, 0.7], "k--", alpha=0.4, label="perfect calibration")
ax.scatter(calib["p_home_mean"], calib["actual_home_win"],
           s=calib["n"] * 1.5, alpha=0.7, label="bucket (size ∝ n)")
for _, row in calib.iterrows():
    ax.annotate(f"{int(row['n'])}", (row["p_home_mean"], row["actual_home_win"]),
                xytext=(6, 6), textcoords="offset points", fontsize=8, alpha=0.6)
ax.axhline(test.home_win.mean(), color="red", alpha=0.3, linestyle=":",
           label=f"empirical home win rate {test.home_win.mean():.3f}")
ax.set_xlabel("predicted p_home (bucket mean)")
ax.set_ylabel("actual home win rate")
ax.set_title("Calibration on 2025 — discrimination spread is the betting signal")
ax.grid(alpha=0.2)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

spread = calib.iloc[-1]["actual_home_win"] - calib.iloc[0]["actual_home_win"]
print(f"\nSpread: bucket 9 actual = {calib.iloc[-1]['actual_home_win']:.3f},  "
      f"bucket 0 actual = {calib.iloc[0]['actual_home_win']:.3f},  "
      f"spread = {spread:+.3f}")
print("That spread is the discriminative signal — and it survives out-of-sample.")
print("The bars sitting ABOVE the diagonal is the 'under-predicts home' bias —")
print("a fixable Platt-scaling problem driven by 2024 having no home-field advantage.")


In [ ]:
# Selective accuracy: the metric that matters most for betting.
# Only bet when the model says >55% home or >55% away — skip the mush.
test_eval = test.copy()
test_eval["p_home"] = p_home
test_eval = test_eval.dropna(subset=["home_win"])

thresholds = [0.50, 0.52, 0.55, 0.58, 0.60]
print("=== Selective accuracy by confidence threshold ===")
print(f"{'thresh':>8} {'bets':>6} {'cover':>6} {'acc':>8} {'home_acc':>9} {'away_acc':>9}")
for thr in thresholds:
    edge = (test_eval["p_home"] - 0.5).abs()
    keep = edge >= (thr - 0.5)
    sub = test_eval[keep]
    if sub.empty:
        continue
    picks = (sub["p_home"] > 0.5).astype(int)
    correct = (picks == sub["home_win"]).mean()
    home_mask = sub["p_home"] > 0.5
    away_mask = ~home_mask
    home_acc = (sub.loc[home_mask, "home_win"] == 1).mean() if home_mask.any() else float("nan")
    away_acc = (sub.loc[away_mask, "home_win"] == 0).mean() if away_mask.any() else float("nan")
    print(f"{thr:>8.2f} {len(sub):>6} {len(sub)/len(test_eval):>6.1%} "
          f"{correct:>8.3%} {home_acc:>9.3%} {away_acc:>9.3%}")

print()
print("Reading: at thr>=0.55 the model bets ~28% of games at ~57% accuracy, and 63.6% on its")
print("home picks. At thr>=0.58 it tightens to ~9% of games at ~59% accuracy. That's the")
print("regime where the model's edge clears the home-base-rate noise and where we'd bet.")


In [ ]:
# Ablation: does more historical data help, or hurt? Train on each candidate
# split, score 2025 the same way, and compare.
#
# This is the most informative experiment in this section: it tells us whether
# the way to improve the model is "more years" or "better features."

splits = {
    "2024 only":          [2024],
    "2023 + 2024":        [2023, 2024],
    "2022 + 2023 + 2024": [2022, 2023, 2024],
}

print(f"{'split':>21}  {'n_train':>8}  {'acc':>7}  {'log_loss':>9}  {'Brier':>7}  "
      f"{'spread':>7}  {'sel acc':>8}  {'sel home':>9}  {'n_bets':>7}")
print("-" * 110)
for label, years in splits.items():
    tr = pd.concat(
        [pd.read_parquet(f"data/features/training_{y}.parquet") for y in years],
        ignore_index=True,
    )
    m = train_runs_model(tr)
    pred = predict_runs(m, test)
    p = pythag_win_prob(pred["home_runs_pred"], pred["away_runs_pred"])
    pm = pythag_metrics(test.home_win, p)
    calib_ = calibration_table(p, test.home_win)
    spread = calib_.iloc[-1].actual_home_win - calib_.iloc[0].actual_home_win
    te = test.copy(); te["p"] = p
    keep = (te["p"] - 0.5).abs() >= 0.05  # threshold = 0.55
    s = te[keep]
    sel_acc = ((s["p"] > 0.5).astype(int) == s.home_win).mean()
    sel_h = (s.loc[s["p"] > 0.5, "home_win"] == 1).mean() if (s["p"] > 0.5).any() else float("nan")
    print(f"{label:>21}  {m.train_n:>8,}  {pm.accuracy:>6.3%}  {pm.log_loss:>9.4f}  "
          f"{pm.brier:>7.4f}  {spread:>+7.3f}  {sel_acc:>7.3%}  {sel_h:>8.3%}  {len(s):>7}")

print()
print("Reading the columns:")
print("  spread    = (top-bucket actual home win rate) - (bottom-bucket actual). Higher = better discrimination.")
print("  sel acc   = accuracy on games where the model's |p_home - 0.5| >= 0.05 (the bets we'd actually take).")
print("  sel home  = accuracy on the SUBSET of those bets where the model picks home.")
print()
print("Result: adding 2022 and 2023 MONOTONICALLY reduces every metric that matters.")
print("Discrimination spread falls from 0.121 -> 0.105 -> 0.093 and home-pick accuracy")
print("at the betting threshold drops from 62.3% -> 60.8% -> 59.8%. The pitch-clock /")
print("shift-restriction regime change in 2023 makes older years less representative of")
print("today's baseball than just having one more year of training data is worth.")
print()
print("Implication for the roadmap: don't go further back than 2024. Instead, INVEST in")
print("features that capture current state (rolling-window stats, park context, defense).")
print("More data with stale features < less data with current-state features.")


### Takeaways from v1

What we learned by training:

- **The features carry real signal.** Even out-of-sample on 2025, the model's top-bucket games show 60% home win rate vs ~50% for the bottom bucket. That spread is the discriminative property a betting model needs.
- **Raw accuracy isn't the right metric.** "Always home" gets 54.3% on 2025 because the home base rate is itself a strong predictor on noisy single games. The model has to do something more nuanced — pick the games where the edge is large enough to bet.
- **The model under-predicts home wins.** Training periods (2023-2024) had near-zero or slightly negative home-team scoring advantage; 2025 has +0.09 runs. The calibration curve sits above the diagonal as a result — fixable post-hoc by Platt scaling on a rolling holdout in production.
- **Coefficients agree with baseball intuition** — once we use the slim feature set. The kitchen-sink set has enough collinearity (SIERA is mathematically a function of K%/BB%/GB%/FB%) that Ridge flips SIERA's sign. The slim default keeps one summary stat per tier.
- **More historical data hurts.** The year-split ablation above is the biggest surprise: 2022-2024 trained on 3× the data still loses to 2024-only across discrimination, selective accuracy, and home-pick accuracy. The 2023 pitch-clock regime change makes pre-2024 years a worse fit for 2025 than no data at all in those years.

### Option B roadmap (what to layer on next)

| Addition | Where it lives | Expected effect |
|---|---|---|
| Park run factor (multiplicative on air-ball share) | `build_features.py` + `matchup.apply_park_adjustment` | Disambiguate Coors / Petco run environments. Few percent on calibration in extreme parks. |
| Team OAA per season | `build_features.py` + `matchup.apply_defense_oaa` | Captures defensive run prevention. ~0.05-0.1 RA/game swing for elite vs. poor defenses. |
| Rolling-window features (30/60-day xwOBA, SIERA) | new column family in `cumulative.py` | Likely the BIGGEST win — captures current form vs. season-cumulative. |
| Rest days / travel | MLB-StatsAPI schedule diff | Small but cheap. |
| Platt scaling / isotonic recalibration | `src/model/evaluate.py` | Fixes the under-prediction-of-home level bias seen above. |
| Continual retraining + rolling holdout | `src/model/...` (cron job) | Production hygiene. |

Each of these slots into the existing pipeline without touching the model layer — add the column, append the column name to `feature_cols`, retrain.


---

## 16. Matchup xwOBA — flight × park × defense

The flat per-hitter park factor from the previous iteration didn't use
the *pitcher* on the mound. This section's adjustment is the full
matchup model. Two steps, both keyed by hitter handedness.

### Step 1 — blend the contact profile

Per decision #4 (the original framework), the expected batted-ball
distribution for a specific (pitcher, hitter) pairing is a 60/40
weighted average favoring the pitcher (because pitchers control launch
angle more than hitters do):

\[\text{blend}_t = 0.6 \cdot p_t^{\text{pitcher}} + 0.4 \cdot p_t^{\text{hitter}}, \quad t \in \{\text{GB, FB, LD, PU}\}\]

### Step 2a — park factor by flight type

Each flight type maps to the venue's Savant park factor for the dominant
hit outcomes of that flight, looked up by hitter handedness. The per-event
factors collapse into per-flight factors via a wOBA-weighted average over
the league-typical event-outcome distribution conditional on flight:

| Flight | Conditional event mix (rough) | Park factor used |
|---|---|---|
| GB | mostly out, some 1B | 1.0 (per user spec — park-neutral) |
| LD | 50% 1B, 15% 2B, 2% 3B, 3% HR | wOBA-weighted blend of (1B, 2B, 3B, HR) factors |
| FB | 2% 1B, 3% 2B, 1% 3B, 9% HR | wOBA-weighted blend (HR-dominated) |
| PU | ~always out | 1.0 |

Then the matchup-level park multiplier is itself a wOBA-weighted average
over flight types, using the blended distribution:

\[\text{PF}_{\text{matchup}} = \frac{\sum_t \text{blend}_t \cdot w^{\text{wOBA}}_t \cdot \text{pf}_t}{\sum_t \text{blend}_t \cdot w^{\text{wOBA}}_t}\]

with league wOBA per BIP type (0.21 GB, 0.34 FB, 0.685 LD, 0.01 PU).

### Step 2b — defense factor by flight type

Defense applies asymmetrically: good infielders suppress GB-share xwOBA;
good outfielders suppress FB- and LD-share xwOBA. The OAA values used
are per-handedness (`outs_above_average_rhh` / `_lhh`) summed across the
**eight non-DH fielders** in the opposing lineup, where each defender's
OAA is looked up by the position they actually played in that
boxscore (DH and pinch-hits contribute zero, catcher OAA isn't published
on Savant):

\[D = 1 - k \cdot \left[\text{IF}_{\text{OAA}} \cdot \text{blend}_{\text{GB}} + \text{OF}_{\text{OAA}} \cdot (\text{blend}_{\text{FB}} + \text{blend}_{\text{LD}})\right]\]

\(k \approx 0.00115\) calibrated so the league-average flight mix
recovers the same total OAA effect (`oaa_runs_per_unit = 0.0011`) used
elsewhere in the matchup engine.

### Final

\[\widetilde{\text{xwOBA}}_{\text{matchup}} = \text{xwOBA} \cdot \text{PF}_{\text{matchup}} \cdot D\]

Done per hitter slot (for the offensive composite) and per opposing-hitter
slot (for the pitcher's expected xwOBA-allowed). The same PF × D
multiplier applies to both — the difference is whether we scale the
hitter's solo xwOBA or the pitcher's solo xwOBA-allowed.

### Data sources

* **Park factors**: `src.ingest.fetch_park_factors` (Savant leaderboard,
  3-year rolling, handedness-split).
* **OAA**: `src.ingest.fetch_oaa` (pybaseball's
  `statcast_outs_above_average`, per-position per-handedness).
* **Lineup positions**: extracted from boxscore JSON via
  `src.features.lineup_loader` — DH/PH/PR slots are properly excluded
  from defense aggregation.

`build_features.py` emits both cumulative and rolling-30d flavors of the
adjusted xwOBA (`*_xwOBA_matchup_adj`, `*_xwOBA_30d_matchup_adj`) for both
sides (offense + opposing starter), so the model layer picks via its
feature list.

In [ ]:
# Sanity check the matchup model on a few extreme cases.
# Three knobs to vary: blended flight profile, venue × handedness, defense.
from src.features.matchup_xwoba import (
    flight_park_factors, matchup_xwoba_adjustment, matchup_park_factor,
)
from src.features.handedness import load_hit_type_park_factor_lookup
from src.features.team_defense import load_oaa, build_oaa_lookup

htl = load_hit_type_park_factor_lookup(
    "data/park_factors/park_factors_2024_rolling3.parquet"
)

print("Per-flight park factors at extreme venues (RHB and LHB):")
print(f'  {"venue/stand":<14}{"GB":>7}{"FB":>7}{"LD":>7}{"PU":>7}')
for v in ("COL", "CIN", "NYY", "BOS", "SEA", "SD", "OAK", "PIT"):
    for s in ("R", "L"):
        pf = flight_park_factors(htl.get((v, s)))
        print(f'    {v}/{s:<10}{pf["GB"]:>7.3f}{pf["FB"]:>7.3f}'
              f'{pf["LD"]:>7.3f}{pf["PU"]:>7.3f}')
print()

# Pure park-only effect (defense = 0): how big is the matchup PF for a
# FB-heavy slugger (Judge-ish) vs a GB-heavy contact bat (Altuve-ish)
# facing typical-profile pitchers at extreme parks?
SCENARIOS = [
    ("FB slugger vs FB pitcher", (0.30, 0.45, 0.20, 0.05),
                                 (0.25, 0.50, 0.22, 0.03)),
    ("GB contact vs GB pitcher", (0.55, 0.25, 0.18, 0.02),
                                 (0.50, 0.25, 0.22, 0.03)),
    ("balanced matchup",         (0.45, 0.35, 0.18, 0.02),
                                 (0.43, 0.35, 0.20, 0.02)),
]
print(f'Matchup PF (xwOBA multiplier, no defense) by venue × matchup × hitter stand:')
print(f'  {"venue/stand":<14}{"matchup":<25}{"PF":>7}')
for v in ("COL", "SEA", "CIN", "OAK"):
    for s in ("R", "L"):
        for name, p, h in SCENARIOS:
            adj, pf, d, blend = matchup_xwoba_adjustment(
                hitter_xwoba=0.375,
                pitcher_flight=p, hitter_flight=h,
                hit_type_pfs=htl.get((v, s)),
                infield_oaa=0, outfield_oaa=0,
            )
            print(f'  {v}/{s:<10}{name:<25}{pf:>7.4f}')
        print()

# Defense effect: vs a +10/+10 OAA opponent vs a -10/-10 OAA opponent.
print("Defense effect (vs. league-typical FB-heavy matchup at neutral park):")
for inf_oaa, of_oaa, label in [(10, 10, '+10 IF / +10 OF (elite D)'),
                                (0, 0, '0 / 0 (avg)'),
                                (-10, -10, '-10 IF / -10 OF (bottom D)')]:
    adj, pf, d, blend = matchup_xwoba_adjustment(
        hitter_xwoba=0.375,
        pitcher_flight=(0.30, 0.45, 0.20, 0.05),
        hitter_flight=(0.25, 0.50, 0.22, 0.03),
        hit_type_pfs={'1B':1.0,'2B':1.0,'3B':1.0,'HR':1.0,'BB':1.0},
        infield_oaa=inf_oaa, outfield_oaa=of_oaa,
    )
    print(f'  {label:<32} Def={d:.4f}  adj_xwOBA={adj:.4f}')

print()
print("Takeaways:")
print("  - Coors RHB: FB and LD both run hot (HR + 2B/3B all elevated)")
print("  - SEA: matchup PF ~0.93 — pitcher park across the board")
print("  - Elite defense subtracts ~2.5% from xwOBA — small per-PA but")
print("    real over a 30-game stretch")

In [ ]:
# Four-way ablation: baseline (park-neutral) vs cumulative+matchup vs
# rolling-30d+matchup vs combined. Same Ridge regression (alpha=1.0)
# everywhere; only the feature set changes.
import pandas as pd
import numpy as np

from src.model.runs_model import (
    train_runs_model, predict_runs, coefficient_table,
    DEFAULT_FEATURE_COLS, BASELINE_FEATURE_COLS,
    ROLLING_FEATURE_COLS, COMBINED_FEATURE_COLS,
)
from src.model.evaluate import (
    runs_metrics, pythag_win_prob, pythag_metrics, calibration_table,
)

train = pd.read_parquet("data/features/training_2024.parquet")
test  = pd.read_parquet("data/features/training_2025.parquet")
test  = test[test.home_score.notna() & test.away_score.notna()].reset_index(drop=True)

def score_set(name, cols):
    m = train_runs_model(train, feature_cols=cols, alpha=1.0)
    pred = predict_runs(m, test)
    p_home = pythag_win_prob(pred.home_runs_pred, pred.away_runs_pred)
    hw = (pred.home_score > pred.away_score).astype(int)
    pm = pythag_metrics(hw, p_home)
    rm_h = runs_metrics(pred.home_score, pred.home_runs_pred)
    rm_a = runs_metrics(pred.away_score, pred.away_runs_pred)
    return {
        "split": name, "n_feat": len(cols),
        "mae": (rm_h.mae + rm_a.mae) / 2,
        "acc": pm.accuracy, "log_loss": pm.log_loss, "Brier": pm.brier,
        "mean_p": pm.mean_p_home, "hw_rate": pm.home_win_rate,
        "_model": m, "_p_home": p_home, "_hw": hw,
    }

results = [
    score_set("baseline (park-neutral)",        BASELINE_FEATURE_COLS),
    score_set("cumulative + matchup_adj",       DEFAULT_FEATURE_COLS),
    score_set("rolling-30d + matchup_adj",      ROLLING_FEATURE_COLS),
    score_set("combined (cum + 30d)",           COMBINED_FEATURE_COLS),
]

summary = pd.DataFrame([
    {k: v for k, v in r.items() if not k.startswith("_")}
    for r in results
])
print("=== Ablation: 2024 -> 2025 (matchup-based xwOBA = flight × park × defense) ===")
print(summary.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
# Inspect coefficients + calibration on the best feature set.
best = max(results, key=lambda r: r["acc"])
print(f'Best set: {best["split"]}  (acc={best["acc"]:.4f}, log_loss={best["log_loss"]:.4f})')
print()
print("Coefficients (standardized scale):")
print(coefficient_table(best["_model"]).to_string(index=False))
print()
print("Calibration (10 buckets, predicted vs actual home win rate):")
print(calibration_table(best["_p_home"], best["_hw"], bins=10).to_string())
print()
print("Calibration gap is positive across every bucket -> the model systematically")
print("under-predicts home wins. Mean predicted p_home ~", round(best["mean_p"], 3),
      "vs actual home win rate ~", round(best["hw_rate"], 3))
print("Platt scaling / isotonic regression is the next fix.")

### Section 16 takeaways

- **Matchup-adjusted xwOBA replaces the flat per-hitter park factor.**
  The flight-blend × per-flight park factor × per-handedness OAA defense
  formula uses the pitcher's flight tendencies and the *actual* eight
  fielders behind him (DH-excluded, position-keyed) instead of just the
  hitter's solo profile.
- **Rolling-30d + matchup_adj is the strongest variant** so far at
  ~53.3% accuracy (+1.4pp over the park-neutral baseline) — a touch
  better than the previous "park-only" matchup adjustment (53.1%).
  Adding the defense layer on top of park buys ~+0.2pp in this 2024->25
  sample.
- **The defense lift is small per-PA but real.** A team with +10/+10
  OAA suppresses xwOBA by ~2.5%; over a 30-game stretch that's a
  meaningful runs delta even though any single PA barely moves.
- **Run-prediction MAE is essentially unchanged across all four sets
  (~2.52).** The accuracy lift is coming from the runs-prediction
  *ranking*, not absolute miss — the matchup adjustment moves a small
  fraction of games across the 0.5 win-probability threshold.
- **The calibration gap is still positive in every bucket** — home
  teams win at a 4.5pp higher clip than the model predicts. This is
  structural (the `is_home` coefficient is being suppressed by other
  features), not a matchup-feature issue. Next step: Platt scaling /
  isotonic regression on a held-out slice.

---

## 17. Sandbox

Free space below. Ideas worth poking at:

- **Recompute FIP / xFIP / SIERA** for the 2025 leaderboard, sort, and compare to your subjective ranking of who pitched well.
- **Build a Statcast aggregate** for one pitcher: K%, BB%, GB%, FB%, LD% from a date-range, then plug into `siera(...)`.
- **Pitch run values for a hitter** vs each pitch type — which hitters punish sliders? fastballs?
- **PullAir% definition:** % of batted balls where `bb_type in ('fly_ball', 'line_drive')` AND `hc_x` is on the batter's pull side. Validate the threshold against league average.
- **Backtest the Pythag formula** on `schedule_and_record` data: feed actual RS / RA for completed games and check the resulting win prob against the actual W/L.

Save anything that looks like a candidate via `save_raw(...)`.
